# Process Documents

This document shows you how to load a pdf from azure blob storage, gain some anaysis and insight with AI and save results to a JSON file for analysis.

## Overview
- Step 1: Install required packages
- Step 2: Connect to AI Foundry
- Step 3: Load in first PDF and extract text
- Step 4: Chunk data (simple page split)
- Step 5: Extract insight from chunk
- Step 6: Process all PDFs and create structured dictionary
- Step 7: Analyze the chunk dictionary
- Step 8: Basic analysis of chunk dictionary

## Step 1: Install Required Packages

First, we'll install the necessary Python packages for this notebook. This includes packages for:
- PDF processing
- Authentication to Azure services
- Foundry SDK
- Data handling

In [ ]:
# Install required packages
%pip install python-dotenv
%pip install azure-identity
%pip install azure-ai-projects
%pip install PyPDF2
%pip install pandas matplotlib
%pip install tiktoken

## Step 2: Connect to AI Foundry

Next, we'll establish a connection to Azure AI Foundry using the authentication helper module. This allows us to use Azure OpenAI services for analyzing our documents.

In [ ]:
# Azure Authentication using Helper Module
import os
from dotenv import load_dotenv
from azure_auth_helper import authenticate_azure

# Load environment variables
load_dotenv("./.env")

# Get tenant ID from environment variables
TENANT_ID = os.getenv('AZURE_TENANT_ID')
if not TENANT_ID:
    raise ValueError("AZURE_TENANT_ID not found in .env file. Please add it to your .env file.")

print(f"🏢 Using tenant ID: {TENANT_ID}")

# Authenticate with Azure using browser authentication
credential = authenticate_azure(
    auth_method='browser', 
    tenant_id=TENANT_ID
)

# Test that the credential works
print(f"🔍 Testing credential type: {type(credential).__name__}")
try:
    # Try to get a token to validate the credential
    token = credential.get_token("https://management.azure.com/.default")
    print("✅ Credential test successful!")
    print(f"Token expires: {token.expires_on}")
except Exception as e:
    print(f"❌ Credential test failed: {e}")
    print(f"   Error type: {type(e).__name__}")
    raise

print("🎉 Ready to use Azure AI Foundry!")

In [ ]:
# Import helper libraries and establish Foundry connection
import numpy as np
import pandas as pd
from azure.ai.projects import AIProjectClient

# Create Foundry project client
project = AIProjectClient(
    endpoint=os.getenv("FOUNDRY_API_ENDPOINT"),
    credential=credential,
)

# Get model deployment names from environment
gpt4o_model = os.getenv('GPT4O_DEPLOYMENT_NAME', 'gpt-4o')

# Now we can use the Foundry project client to get the OpenAI client
# This gives us all the Foundry benefits like monitoring and tracking
models = project.get_openai_client(api_version="2024-10-21")

print("✅ Successfully connected to Azure AI Foundry")
print(f"📊 Project endpoint: {os.getenv('FOUNDRY_API_ENDPOINT')}")
print(f"🏢 Resource Group: {os.getenv('AZURE_RESOURCE_GROUP')}")
print(f"🤖 Using model: {gpt4o_model}")
print("🔑 Using browser credential for project authentication")

## Step 3: Load PDF and Extract Text

In this step, we'll load our first PDF document from the data directory and extract its text content. We'll use the PyPDF2 library to parse the PDF file.

In [ ]:
# Import necessary libraries for PDF processing
import PyPDF2
import os
import glob
from pathlib import Path

# Define a function to extract text from PDF files
def extract_text_from_pdf(pdf_path):
    """
    Extract text from a PDF file and return a dictionary with page number and text.
    """
    pdf_text = {}
    
    try:
        with open(pdf_path, 'rb') as pdf_file:
            # Create PDF reader object
            pdf_reader = PyPDF2.PdfReader(pdf_file)
            
            # Get number of pages
            num_pages = len(pdf_reader.pages)
            print(f"📄 Processing PDF with {num_pages} pages")
            
            # Extract text from each page
            for page_num in range(num_pages):
                page = pdf_reader.pages[page_num]
                pdf_text[page_num + 1] = page.extract_text()
                
            print(f"✅ Successfully extracted text from all {num_pages} pages")
            
    except Exception as e:
        print(f"❌ Error extracting text from PDF: {e}")
        raise
        
    return pdf_text

# Path to our first PDF file
pdf_path = os.path.join("data", "brownfield_redevelopment_report.pdf")
print(f"📁 Loading PDF from: {pdf_path}")

# Extract text from the PDF
pdf_text = extract_text_from_pdf(pdf_path)

# Display information about the extracted text
print(f"📊 Extracted text from {len(pdf_text)} pages")
print(f"📝 First page sample: {pdf_text[1][:500]}...")

## Step 4: Chunk Data (Simple Page Split)

Now that we have extracted the text from our PDF, we'll organize it into chunks for processing. For simplicity, we'll use each page as a separate chunk. But this 

In [ ]:
# Define a function to organize PDF text into chunks
def chunk_pdf_text(pdf_text):
    """
    Organize PDF text into chunks, with each page as a chunk.
    Returns a list of dictionaries with page number and text content.
    """
    chunks = []
    
    for page_num, text in pdf_text.items():
        # If the text is too short (likely an empty page or just a heading), skip it
        if len(text.strip()) < 50:
            continue
            
        chunks.append({
            "page_number": page_num,
            "content": text,
            "source": "brownfield_redevelopment_report.pdf"
        })
        
    return chunks

# Chunk the PDF text
pdf_chunks = chunk_pdf_text(pdf_text)

# Display information about chunks
print(f"🧩 Created {len(pdf_chunks)} chunks from the PDF")

# Display first chunk as an example
print("\n📝 First chunk sample:")
print(f"Page: {pdf_chunks[0]['page_number']}")
print(f"Content: {pdf_chunks[0]['content'][:150]}...")

## Step 5: Extract Insights from Chunks (Initial Test)

In this step, we'll use the Azure AI Foundry OpenAI model to extract insights from text. As an initial test, we'll only process the first chunk from our first document to verify our approach works correctly before applying it to all documents in the next step. This test helps us validate the extraction function and response format before running it on all chunks and documents.

In [ ]:
# Define a function to extract insights from text using Azure OpenAI
def extract_insights_from_chunk(chunk_content, model_client, model_name, topics_to_identify=None, topics_to_summarize=None):
    """
    Extract structured insights from a chunk of text using Azure OpenAI.
    Returns a dictionary with the extracted insights.
    
    Parameters:
    - chunk_content: The text content to analyze
    - model_client: The Azure OpenAI client
    - model_name: The name of the model to use
    - topics_to_identify: List of topics to identify in the text (broader list)
    - topics_to_summarize: List of topics to create detailed summaries for (subset of topics_to_identify)
    """
    # Define default topics if none provided
    if topics_to_identify is None:
        topics_to_identify = ["Water Quality", "Climate Change", "Biodiversity", "Waste Management", 
                             "Social Impact", "Air Quality", "Energy Use", "Transportation", "Land Use"]
    
    # If topics to summarize not provided, use an empty list (identify only)
    if topics_to_summarize is None:
        topics_to_summarize = []
    
    # Create formatted strings for the prompt
    identify_list = ", ".join([f'"{topic}"' for topic in topics_to_identify])
    summarize_list = ", ".join([f'"{topic}"' for topic in topics_to_summarize])
    
    system_prompt = f"""
    You are an expert environmental analyst. Extract key sustainability insights from the provided text.
    
    PART 1: First, identify which of the following topics are SUBSTANTIALLY discussed in the text: {identify_list}.
    
    IMPORTANT: Only include topics that are discussed in detail with meaningful information. DO NOT include topics 
    that are merely mentioned in passing or only briefly referenced without substantial content.
    
    PART 2: For the following subset of topics (if present in the text), provide a detailed summary:
    {summarize_list if topics_to_summarize else "No topics require detailed summaries."}
    
    Format your response as a JSON object with these fields:
    {{
        "identified_topics": ["Topic1", "Topic2", ...],  // Only topics discussed in detail from Part 1
        "topic_summaries": {{
            // Only include summaries for topics from Part 2 that were actually identified
            "Topic1": "Detailed summary of information about Topic1",
            "Topic2": "Detailed summary of information about Topic2",
            ...
        }},
        "additional_topics": ["Other Topic1", "Other Topic2", ...],  // Topics not in the provided list but substantially discussed in text
        "general_summary": "A 1-2 sentence general summary of this section"
    }}
    """
    
    user_prompt = f"Extract insights from the following text:\n\n{chunk_content}"
    
    try:
        # Call Azure OpenAI through Foundry
        response = model_client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0,  # Use low temperature for deterministic results
            response_format={"type": "json_object"}  # Request JSON format
        )
        
        # Extract and parse the response
        insights_text = response.choices[0].message.content
        
        # Return the insights as a string (we'll parse it later)
        return insights_text
        
    except Exception as e:
        print(f"❌ Error extracting insights: {e}")
        return None

# Define topics to identify (broader list)
topics_to_identify = [
    "Water Quality", 
    "Climate Change", 
    "Biodiversity", 
    "Soil Remediation",
    "Air Quality", 
    "Waste Management",
    "Social Equity",
    "Energy Efficiency",
    "Transportation",
    "Land Use",
    "Noise Pollution",
    "Cultural Heritage",
    "Urban Agriculture",
    "Affordable Housing"
]

# Define topics to summarize (focused subset)
topics_to_summarize = [
    "Water Quality", 
    "Climate Change", 
    "Waste Management",
    "Mammal Impact"
]

# Process only the first chunk as a test
print("🔍 TESTING ONLY: Extracting insights from the first chunk...")
print("   (In Step 6, we'll process all chunks from all documents)")

first_chunk_insights = extract_insights_from_chunk(
    pdf_chunks[0]["content"], 
    models, 
    gpt4o_model,
    topics_to_identify,
    topics_to_summarize
)

print(f"✅ Successfully extracted insights from test chunk")
print(f"\n📝 Insights from first chunk (test):")
print(first_chunk_insights)

## Step 6: Process All PDF Files and Create Structured Dictionary

Now that we've verified our extraction process works with a single chunk, we'll apply it to all PDF files in our data directory and create a structured dictionary of the results. This includes:

1. Processing all PDFs in the data directory (including the first one we tested in Step 5)
2. Processing all chunks (pages) from each document
3. Extracting insights using the same function we tested in Step 5
4. Creating a dictionary entry for each chunk with original text, page number, document name, topics, and summaries
5. Saving the structured data as a JSON file

In [ ]:
# Process all PDF files and create a structured dictionary
import json
import time

# List all PDF files in the data directory
pdf_files = glob.glob(os.path.join("data", "*.pdf"))
print(f"📚 Found {len(pdf_files)} PDF files in the data directory:")
for pdf_file in pdf_files:
    print(f"  - {os.path.basename(pdf_file)}")

# List to store all chunks with their insights
all_chunks = []

# Process each PDF file
print("\n🔄 Starting complete analysis of all documents (including first document from Step 5)")

for pdf_file in pdf_files:
    filename = os.path.basename(pdf_file)
    print(f"\n📄 Processing {filename}...")
    
    # Extract text from PDF
    pdf_text = extract_text_from_pdf(pdf_file)
    
    # Create chunks from all pages
    chunks = chunk_pdf_text(pdf_text)
    print(f"   Created {len(chunks)} chunks from {len(pdf_text)} pages")
    
    # Process each chunk
    for i, chunk in enumerate(chunks):
        print(f"  🔍 Processing chunk {i+1}/{len(chunks)} (page {chunk['page_number']})...")
        
        # Extract insights with our topics of interest
        insights_text = extract_insights_from_chunk(
            chunk["content"], 
            models, 
            gpt4o_model,
            topics_to_identify,
            topics_to_summarize
        )
        
        # Parse JSON insights
        try:
            insights = json.loads(insights_text)
            
            # Create a dictionary entry for this chunk
            chunk_entry = {
                "document": filename,
                "page_number": chunk["page_number"],
                "original_text": chunk["content"],
                "topics": insights.get("identified_topics", []),
                "topic_summaries": insights.get("topic_summaries", {}),
                "additional_topics": insights.get("additional_topics", []),
                "general_summary": insights.get("general_summary", "")
            }
            
            # Add to the list of all chunks
            all_chunks.append(chunk_entry)
            
            # Print identified topics
            identified_topics = insights.get("identified_topics", [])
            summarized_topics = list(insights.get("topic_summaries", {}).keys())
            
            print(f"    ✅ Identified topics (discussed in detail): {', '.join(identified_topics) if identified_topics else 'None'}")
            if summarized_topics:
                print(f"    📝 Summarized topics: {', '.join(summarized_topics)}")
                
        except json.JSONDecodeError:
            print(f"    ❌ Error parsing insights JSON")
            
        # Add a short delay between API calls to avoid rate limiting
        time.sleep(0.5)

# Print completion message
print(f"\n✅ Completed full analysis: Processed {len(pdf_files)} documents with {len(all_chunks)} total chunks")

# Save all chunks to a JSON file
json_file_path = "document_chunks.json"
with open(json_file_path, 'w') as json_file:
    json.dump(all_chunks, json_file, indent=2)

print(f"\n💾 Saved all {len(all_chunks)} chunks to {json_file_path}")

# Display an example of a chunk entry
if all_chunks:
    print("\n📝 Example - First chunk entry:")
    print(f"Document: {all_chunks[0]['document']}")
    print(f"Page number: {all_chunks[0]['page_number']}")
    print(f"Topics: {all_chunks[0]['topics']}")
    print(f"Topic summaries: {len(all_chunks[0]['topic_summaries'])}")
    print(f"Text preview: {all_chunks[0]['original_text'][:100]}...")